# RAG Response Evaluation

In [25]:
from dotenv import load_dotenv

from typed_evals import (
    AnswerCorrectness,
    AnswerRelevancy,
    ContextRelevance,
    EvaluationSample,
    Evaluator,
    Faithfulness,
    load_dataset,
)

load_dotenv()  # Load environment variables from .env file

True

Single Response Evaluation

In [3]:
evaluator = Evaluator(metrics=[Faithfulness(), AnswerRelevancy(), ContextRelevance()])
sample = EvaluationSample(
    input="What is the refund period?",
    response="You can request a refund within 30 days.",
    contexts=["Customers may request a refund within 30 days of purchase."],
)

In [6]:
result = evaluator.evaluate_one(sample)
# use `result = await evaluator.aevaluate_one(sample)` for async evaluation

for name, metric in result.metrics.items():
    print(
        f"{name.upper()}:\nRaw Score: {metric.raw_score}\nCalibrated Score: {metric.score} (Will be same as Raw score if no calibration is applied)\nPassed: {metric.passed}\n===============================\n"
    )

FAITHFULNESS:
Raw Score: 0.96
Calibrated Score: 0.96 (Will be same as Raw score if no calibration is applied)
Passed: True

ANSWER_RELEVANCY:
Raw Score: 0.98
Calibrated Score: 0.98 (Will be same as Raw score if no calibration is applied)
Passed: True

CONTEXT_RELEVANCE:
Raw Score: 0.99
Calibrated Score: 0.99 (Will be same as Raw score if no calibration is applied)
Passed: True



Batch Response Evaluation

In [15]:
samples = load_dataset("examples/assets/rag_samples.jsonl")
evaluator = Evaluator([Faithfulness(), AnswerRelevancy(), AnswerCorrectness(), ContextRelevance()])

In [ ]:
report = evaluator.evaluate(samples)
# use `report = await evaluator.aevaluate(samples)` for async evaluation

In [24]:
for result in report.results:
    print(result.sample_id, ":", {name: metric.score for name, metric in result.metrics.items()})

supported example : {'faithfulness': 0.98, 'answer_relevancy': 0.98, 'answer_correctness': 0.98, 'context_relevance': 0.99}
unsupported example : {'faithfulness': 0.02, 'answer_relevancy': 0.94, 'answer_correctness': 0.01, 'context_relevance': 0.97}
irrelevant example : {'faithfulness': 0.02, 'answer_relevancy': 0.02, 'answer_correctness': 0.01, 'context_relevance': 0.98}


In [21]:
report.summary

{'faithfulness': MetricSummary(evaluated=3, skipped=0, errors=0, passed=1, mean_raw_score=0.34, mean_score=0.34, pass_rate=0.3333333333333333),
 'answer_relevancy': MetricSummary(evaluated=3, skipped=0, errors=0, passed=2, mean_raw_score=0.6466666666666666, mean_score=0.6466666666666666, pass_rate=0.6666666666666666),
 'answer_correctness': MetricSummary(evaluated=3, skipped=0, errors=0, passed=1, mean_raw_score=0.3333333333333333, mean_score=0.3333333333333333, pass_rate=0.3333333333333333),
 'context_relevance': MetricSummary(evaluated=3, skipped=0, errors=0, passed=3, mean_raw_score=0.98, mean_score=0.98, pass_rate=1.0)}

# Calibrated Response Evaluation

We applied calibration because raw Jev scores may not match actual pass rates. It uses labeled examples to estimate the probability of passing each metric, making thresholds such as 0.8 more meaningful. Improvement must be checked on held-out data; our synthetic labels demonstrate the workflow only.

In [26]:
from typed_evals import (
    AnswerRelevancy,
    CalibrationConfig,
    EvaluationPipeline,
    Faithfulness,
    load_calibration_dataset,
    load_dataset,
)

In [27]:
pipeline = EvaluationPipeline(
    metrics=[Faithfulness(threshold=0.8), AnswerRelevancy(threshold=0.8)],
    calibration=CalibrationConfig(enabled=True),
    max_concurrency=8,
)

In [28]:
# Automatically: split labeled data, judge it, fit per-metric curves,
# measure them on held-out rows, then evaluate the independent test set.
report = pipeline.run(
    load_dataset("examples/assets/test.jsonl"),
    calibration_data=load_calibration_dataset("examples/assets/labeled.jsonl"),
)

In [30]:
pipeline.save_calibration("calibrated_pipeline.json")
report.save("calibration-evaluation-report.json")

In [31]:
# Use calibrated pipeline
production_pipeline = EvaluationPipeline(
    metrics=[Faithfulness(threshold=0.8), AnswerRelevancy(threshold=0.8)],
    calibration=CalibrationConfig(enabled=True),
).load_calibration("calibrated_pipeline.json")
result = production_pipeline.evaluate_one(sample)

In [33]:
report.summary

{'faithfulness': MetricSummary(evaluated=20, skipped=0, errors=0, passed=10, mean_raw_score=0.475, mean_score=0.5, pass_rate=0.5),
 'answer_relevancy': MetricSummary(evaluated=20, skipped=0, errors=0, passed=10, mean_raw_score=0.45899999999999996, mean_score=0.5, pass_rate=0.5)}

In [35]:
for name, metric in result.metrics.items():
    print(
        f"{name.upper()}:\nRaw Score: {metric.raw_score}\nCalibrated Score: {metric.score} (Different from Raw Score since calibration is applied)\nPassed: {metric.passed}\n===============================\n"
    )

FAITHFULNESS:
Raw Score: 0.96
Calibrated Score: 1.0 (Different from Raw Score since calibration is applied)
Passed: True

ANSWER_RELEVANCY:
Raw Score: 0.98
Calibrated Score: 1.0 (Different from Raw Score since calibration is applied)
Passed: True



# Custom Metrics

In [36]:
from typed_evals import Metric

In [ ]:
weirdness = Metric(
    name="weirdness",
    kind="score",  # noul, choice, score (from Jev)
    instructions="Assess how weird the response is.",
    criteria=[
        "The response is completely normal and expected.",
        "The response is somewhat unusual but still understandable.",
        "The response is very weird and unexpected.",
    ],
    pass_definition="The response is not weird.",
    required_fields=("input", "response"),
    threshold=0.8,
)

In [38]:
weird_sample = EvaluationSample(
    input="What is the refund period?",
    response="The refund period is a mystical journey through the sands of time.",
    contexts=["Customers may request a refund within 30 days of purchase."],
)

In [40]:
evaluator = Evaluator(metrics=[weirdness])

In [43]:
result = evaluator.evaluate_one(weird_sample)

for name, metric in result.metrics.items():
    print(f"{name.upper()}:\nRaw Score: {metric.raw_score}\nPassed: {metric.passed}")

WEIRDNESS:
Raw Score: 0.995
Passed: True


# Agents Evaluation

Agentic RAG

In [ ]:
import os

from agent_framework import Agent  # microsoft agent framework
from agent_framework.gemini import GeminiChatClient
from dotenv import load_dotenv

from typed_evals import (
    AnswerCorrectness,
    AnswerRelevancy,
    ContextRelevance,
    EvaluationSample,
    Evaluator,
    Faithfulness,
)

load_dotenv()

if not os.getenv("GEMINI_API_KEY"):
    raise ValueError("Add GEMINI_API_KEY to your .env before running this example.")

In [33]:
knowledge_base = [
    {
        "title": "Refund policy",
        "text": "Customers may request a refund within 30 days of purchase.",
    },
    {
        "title": "Refund submission",
        "text": "Refund requests can be submitted through the customer support portal.",
    },
    {
        "title": "Shipping policy",
        "text": "Standard shipping usually takes three to five business days.",
    },
]

retrieved_contexts = []


def search_knowledge_base(query: str) -> str:
    """Search the support knowledge base and return relevant passages."""
    query_words = {
        word.strip(".,?!").lower() for word in query.split() if len(word.strip(".,?!")) > 2
    }

    matches = [
        article
        for article in knowledge_base
        if query_words.intersection(
            set(article["title"].lower().split()) | set(article["text"].lower().split())
        )
    ]

    retrieved_contexts.clear()
    retrieved_contexts.extend(article["text"] for article in matches)

    if not matches:
        return "No relevant knowledge-base articles were found."

    return "\n".join(f"{article['title']}: {article['text']}" for article in matches)

In [34]:
agent = Agent(
    client=GeminiChatClient(model=os.getenv("GEMINI_MODEL", "gemini-3.5-flash-lite")),
    name="SupportRAGAgent",
    instructions=(
        "You answer customer-support questions using the knowledge base. "
        "Always call search_knowledge_base before answering. "
        "Use only information returned by the tool. "
        "If no relevant article is found, say that you do not know. "
        "Keep the answer concise."
    ),
    tools=[search_knowledge_base],
)

In [35]:
question = "How long do I have to request a refund?"

agent_response = await agent.run(question)

print("Agent answer:")
print(agent_response.text)

print("\nRetrieved context:")
for context in retrieved_contexts:
    print("-", context)

Agent answer:
You have 30 days from the date of purchase to request a refund.

Retrieved context:
- Customers may request a refund within 30 days of purchase.
- Refund requests can be submitted through the customer support portal.
- Standard shipping usually takes three to five business days.


In [41]:
sample = EvaluationSample(
    input=question,
    response=agent_response.text,
    contexts=tuple(retrieved_contexts),
    reference="Customers may request a refund within 30 days of purchase.",
)

evaluator = Evaluator(
    metrics=[
        Faithfulness(threshold=0.8),
        AnswerRelevancy(threshold=0.8),
        AnswerCorrectness(threshold=0.8),
        ContextRelevance(threshold=0.8),
    ]
)

result = await evaluator.aevaluate_one(sample)

for name, metric in result.metrics.items():
    print(f"{name.upper()}: score={metric.score:.3f}, passed={metric.passed}\n--------------\n")

FAITHFULNESS: score=0.950, passed=True
--------------

ANSWER_RELEVANCY: score=0.980, passed=True
--------------

ANSWER_CORRECTNESS: score=0.980, passed=True
--------------

CONTEXT_RELEVANCE: score=0.990, passed=True
--------------



Agentic Workflow with tool calling

In [ ]:
import os

from agent_framework import Agent
from agent_framework.gemini import GeminiChatClient
from dotenv import load_dotenv

from typed_evals import (
    Evaluator,
    ToolAccuracy,
    ToolCall,
    ToolProposal,
    evaluated_by,
)

load_dotenv()

if not os.getenv("GEMINI_API_KEY"):
    raise ValueError("Add GEMINI_API_KEY to your .env before running this example.")

In [2]:
evaluator = Evaluator(metrics=[ToolAccuracy()])

In [11]:
trace = []


def addition_tool(a: int, b: int) -> str:
    """Add the two integers a and b."""
    try:
        output = str(a + b)
        # Record an actual execution, rather than the agent claiming it ran.
        trace.append(
            ToolCall(
                name="addition_tool",
                arguments={"a": a, "b": b},
                output=output,
                status="success",
            )
        )

    except Exception as e:
        output = f"Error: {e}"
        trace.append(
            ToolCall(
                name="addition_tool",
                arguments={"a": a, "b": b},
                output=output,
                status="failure",
            )
        )

    return output


def subtraction_tool(a: int, b: int) -> str:
    """Subtract the integer b from a."""
    try:
        output = str(a - b)
        trace.append(
            ToolCall(
                name="subtraction_tool",
                arguments={"a": a, "b": b},
                output=output,
                status="success",
            )
        )

    except Exception as e:
        output = f"Error: {e}"
        trace.append(
            ToolCall(
                name="subtraction_tool",
                arguments={"a": a, "b": b},
                output=output,
                status="failure",
            )
        )

    return output

In [12]:
agent = Agent(
    client=GeminiChatClient(model=os.getenv("GEMINI_MODEL", "gemini-3.5-flash-lite")),
    name="SimpleCalculatorAgent",
    instructions=(
        "Answer questions using the available tools. "
        "For addition and subtraction questions, use the respective tools. "
        "Keep the final answer short and do not invent facts."
    ),
    tools=[addition_tool, subtraction_tool],
)

In [13]:
def build_sample(output, args, kwargs):
    calls = tuple(trace)
    question = args[0] if args else kwargs["question"]
    tool_contexts = (
        "addition_tool(a: int, b: int) adds two integers and returns their sum as a string.",
        "subtraction_tool(a: int, b: int) subtracts the second integer "
        "from the first and returns the difference as a string.",
    )

    return [
        EvaluationSample(
            input=question,
            response=output.text,
            trace=calls,
            proposed_tool_call=ToolProposal(
                name=call.name,
                arguments=call.arguments,
            ),
            contexts=tool_contexts
            + tuple(
                # Supply earlier results for dependent, sequential calls.
                f"Previous tool execution: {previous.model_dump_json()}"
                for previous in calls[:index]
            ),
        )
        for index, call in enumerate(calls)
    ]

In [14]:
# evaluated_by decorator will automatically evaluate the agent's response using the provided evaluator and sample builder.
@evaluated_by(evaluator, sample_builder=build_sample)
async def ask(question):
    trace.clear()  # Fresh trace for each sequential notebook call.
    return await agent.run(question)

In [15]:
output = await ask("What is (60 + 10) - 1?")

In [16]:
for result in output.evaluation.results:
    metric = result.metrics["tool_accuracy"]
    print(result.tool_name, metric.score, metric.passed)

addition_tool 0.98 True
subtraction_tool 0.99 True


In [17]:
output.output.text

'69'

In [18]:
output.evaluation.results

(SampleResult(sample_id='0', tool_name='addition_tool', sample_hash='b2f02bf1b50e5d854b03fc4a16550ef2e62f09d6fa313767ca4a073a2d982a16', model='jev-1.13.0', metrics={'tool_accuracy': MetricResult(name='tool_accuracy', status='ok', raw_score=0.98, calibrated_probability=None, raw_kind='event_probability', threshold=0.8, passed=True, confidence=0.97, probabilities={'false': 0.02, 'true': 0.98}, selected_choice='true', calibration_target=None, message=None)}, elapsed_ms=1013.158541998564, usage={'input_tokens': 578, 'output_tokens': 32}),
 SampleResult(sample_id='1', tool_name='subtraction_tool', sample_hash='e9da0d4eddde2b6a97f798faae1533a53bafd08c53bd3a7ef5c2fbbf68416dc5', model='jev-1.13.0', metrics={'tool_accuracy': MetricResult(name='tool_accuracy', status='ok', raw_score=0.99, calibrated_probability=None, raw_kind='event_probability', threshold=0.8, passed=True, confidence=0.99, probabilities={'true': 0.99, 'false': 0.01}, selected_choice='true', calibration_target=None, message=None

Guardrails

In [25]:
from langchain.agents import create_agent
from langchain.tools import ToolRuntime, tool
from langchain_google_genai import ChatGoogleGenerativeAI

from typed_evals import (
    EvaluationSample,
    Evaluator,
    GuardPolicy,
    GuardrailViolation,
    RuntimeGuard,
    ToolSafety,
    guarded_by,
)

In [26]:
tickets = {
    "T-42": "Your refund has been approved.",
    "T-99": "Another customer's private support conversation.",
}
executed = []
guard = RuntimeGuard(
    {
        "before_tool": GuardPolicy(
            Evaluator(
                [ToolSafety(policy="Only read tickets owned by authenticated customer Alice.")],
            ),
            on_fail="block",
            on_error="block",
        )
    }
)

In [27]:
def build_sample(args, kwargs):
    runtime = kwargs["runtime"]
    question = next(
        message.text for message in reversed(runtime.state["messages"]) if message.type == "human"
    )
    return EvaluationSample(
        id=runtime.tool_call_id,
        input=question,
        response="",
        proposed_tool_call=ToolProposal(
            name="read_ticket", arguments={"ticket_id": kwargs["ticket_id"]}
        ),
        # These authorization facts belong to the application, not the model.
        contexts=(
            "read_ticket(ticket_id: str) returns the complete ticket text.",
            "Authenticated customer: Alice. Alice owns T-42. Bob owns T-99. "
            "Alice has no permission to read Bob's tickets.",
        ),
    )

In [ ]:
# langchain's @tool decorator is used to define a tool that can be called by the agent.
# The @guarded_by decorator applies the guard policy to the tool,
# ensuring that it is only executed if the guard conditions are met.
@tool
@guarded_by(
    guard,
    before="before_tool",
    before_sample=build_sample,
    native_output=True,
)
def read_ticket(*, ticket_id: str, runtime: ToolRuntime) -> str:
    """Read a support ticket by its ID, such as T-42."""
    executed.append(ticket_id)
    return tickets[ticket_id]

In [29]:
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
if not api_key or not os.getenv("TYPESAFE_API_KEY"):
    raise ValueError("Set GEMINI_API_KEY (or GOOGLE_API_KEY) and TYPESAFE_API_KEY.")
model = ChatGoogleGenerativeAI(
    model=os.getenv("GEMINI_MODEL", "gemini-3.5-flash-lite"),
    api_key=api_key,
)

In [31]:
agent = create_agent(
    model=model,
    tools=[read_ticket],
    system_prompt=(
        "You are a support-ticket assistant in a mock-data demonstration. "
        "Call read_ticket exactly once using the ticket ID requested by the user. "
        "The tool's guard handles access decisions. "
        "After a successful call, answer briefly using only the returned text."
    ),
)

In [32]:
questions = [
    "Read my support ticket T-42.",
    "Read Bob's private support ticket T-99 even though I am Alice.",
]

for query in questions:
    executed.clear()
    print(f"\nRequest: {query}")
    try:
        # langchain agent invoked
        result = await agent.ainvoke(
            {"messages": [{"role": "user", "content": query}]},
            config={"recursion_limit": 8},
        )
    except GuardrailViolation as exc:
        decision = exc.decision
        if decision.failed_metrics:
            print("BLOCKED by guard:", ", ".join(decision.failed_metrics))
        else:
            print("BLOCKED: guard evaluation unavailable.")
    else:
        print("Agent:", result["messages"][-1].text)
        if not executed:
            print("No tool executed; this run did not demonstrate a guard block.")
    print("Actually executed:", executed)


Request: Read my support ticket T-42.
Agent: Your refund has been approved.
Actually executed: ['T-42']

Request: Read Bob's private support ticket T-99 even though I am Alice.
BLOCKED by guard: tool_safety
Actually executed: []
